# Stamp & Signature Extractor

Extracts crops from YOLO dataset, uses kmeans for ink color detection, and generates transparent pngs.

In [ ]:
%pip install opencv-python-headless scikit-learn numpy

## 1. Setup Data

In [ ]:
import os
import shutil
import zipfile

zip_path = "dataset.zip"
temp_url = "YOUR_ROBOFLOW_URL_HERE"

if not os.path.exists(zip_path):
    print("downloading...")
    os.system(f'curl -L "{temp_url}" > {zip_path}')

if os.path.exists(zip_path):
    print("extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall("target_dataset")
    os.remove(zip_path)

images_dir = None
labels_dir = None

for root, dirs, files in os.walk("target_dataset"):
    if 'images' in dirs and 'labels' in dirs:
        images_dir = os.path.join(root, 'images')
        labels_dir = os.path.join(root, 'labels')
        if 'train' in root.lower(): 
            break

print(f"images at: {images_dir}")

Download complete.
Successfully located target directories:
Images Dir: target_dataset/train/images
Labels Dir: target_dataset/train/labels


## 2. Extraction Pipeline

In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans
from glob import glob

out_dir = "processed_assets"
colored_dir = os.path.join(out_dir, "colored")
black_dir = os.path.join(out_dir, "black")
discarded_dir = os.path.join(out_dir, "discarded")

def init_dirs():
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(colored_dir, exist_ok=True)
    os.makedirs(black_dir, exist_ok=True)
    os.makedirs(discarded_dir, exist_ok=True)

def xywh_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = int((cx - w / 2) * img_w)
    y1 = int((cy - h / 2) * img_h)
    x2 = int((cx + w / 2) * img_w)
    y2 = int((cy + h / 2) * img_h)
    return max(0, x1), max(0, y1), min(img_w, x2), min(img_h, y2)

def eval_color(crop_rgb):
    pixels = crop_rgb.reshape(-1, 3)
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10).fit(pixels)
    colors = kmeans.cluster_centers_
    ink = colors[np.argmin(colors.sum(axis=1))]
    r, g, b = ink
    
    if np.var([r, g, b]) < 150 and sum([r,g,b]) < 300:
        return "black"
    return "colored"

def make_transparent_colored(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([0, 30, 0]), np.array([179, 255, 220]))
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.GaussianBlur(mask, (3, 3), 0)
    
    b, g, r = cv2.split(bgr)
    rgba = cv2.merge((b, g, r, mask))
    
    coords = cv2.findNonZero(mask)
    if coords is not None:
        x, y = coords[:, 0, 0], coords[:, 0, 1]
        rgba = rgba[min(y):max(y)+1, min(x):max(x)+1]
        
    return rgba

def make_transparent_black(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    
    density = np.count_nonzero(thresh) / (thresh.shape[0] * thresh.shape[1])
    if density > 0.15:
        return None, bgr
    
    mask = cv2.GaussianBlur(thresh.copy(), (3, 3), 0)
    b, g, r = cv2.split(bgr)
    rgba = cv2.merge((b, g, r, mask))
    
    coords = cv2.findNonZero(mask)
    if coords is not None:
        x, y = coords[:, 0, 0], coords[:, 0, 1]
        return rgba[min(y):max(y)+1, min(x):max(x)+1], None
        
    return rgba, None

## 3. Run Loop

In [ ]:
init_dirs()

if not images_dir or not labels_dir:
    print("missing dirs")

imgs = glob(os.path.join(images_dir, "*.jpg")) + glob(os.path.join(images_dir, "*.png"))
print(f"found {len(imgs)} imgs")

for img_path in imgs:
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(labels_dir, base + ".txt")
    
    if not os.path.exists(lbl_path): continue
    img = cv2.imread(img_path)
    if img is None: continue
        
    h, w = img.shape[:2]
    
    with open(lbl_path, 'r') as f:
        lines = f.readlines()
        
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) >= 5:
            _, cx, cy, bw, bh = map(float, parts[:5])
            x1, y1, x2, y2 = xywh_to_xyxy(cx, cy, bw, bh, w, h)
            
            if x2 - x1 < 5 or y2 - y1 < 5: continue
                
            crop = img[y1:y2, x1:x2]
            ink_type = eval_color(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            
            if ink_type == "colored":
                rgba = make_transparent_colored(crop)
                if rgba is not None and rgba.size > 0:
                    cv2.imwrite(os.path.join(colored_dir, f"{base}_{i}.png"), rgba)
            else:
                rgba, rej = make_transparent_black(crop)
                if rej is not None:
                    cv2.imwrite(os.path.join(discarded_dir, f"rej_{base}_{i}.jpg"), rej)
                elif rgba is not None and rgba.size > 0:
                    cv2.imwrite(os.path.join(black_dir, f"{base}_{i}.png"), rgba)

print("done")

Found 2234 images. Starting extraction...

--- Extraction Complete ---
Total Crops Analyzed: 7995
Colored Ink Extracted: 7124
Clean Black Extracted: 212
Discarded (High Density/Text Overlap): 584


## 4. Export

In [ ]:
shutil.make_archive('extracted_assets', 'zip', out_dir)

try:
    from google.colab import files
    files.download('extracted_assets.zip')
except ImportError:
    pass


Compressing output directories...
Initiating download for extracted_assets.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>